# Step 1 — Download country PBF and filter by boundary

This notebook does two things:

1. **Download** the country OSM PBF from [Geofabrik](https://download.geofabrik.de/) if not already cached in `map/`
2. **Filter** it to your boundary polygon using `osmium extract`, producing a small regional PBF

**Caching:** the country PBF (which can be several hundred MB) is downloaded only once
and reused for all future boundaries in the same country.

**Prerequisites:**
- `osmium-tool` installed: `sudo apt install osmium-tool` / `brew install osmium-tool`
- A boundary GeoJSON in `boundaries/` (from notebook 0)

**Geofabrik URL format:**
```
https://download.geofabrik.de/{continent}/{country}-latest.osm.pbf

Examples:
  europe/sweden   → https://download.geofabrik.de/europe/sweden-latest.osm.pbf
  europe/germany  → https://download.geofabrik.de/europe/germany-latest.osm.pbf
  north-america/us/california → https://download.geofabrik.de/north-america/us/california-latest.osm.pbf
```
Browse all available regions at **https://download.geofabrik.de/**

In [1]:
%%time
import subprocess
import requests
from pathlib import Path
import geopandas as gpd
import folium

# ── Configuration — edit these ────────────────────────────────────────────
COUNTRY_URL   = 'https://download.geofabrik.de/europe/sweden-latest.osm.pbf'
BOUNDARY_NAME = 'nacka' #'sodermalm'   # name of your boundary file in boundaries/ (without .geojson)
# ─────────────────────────────────────────────────────────────────────────

MAP_DIR      = Path('../map');  MAP_DIR.mkdir(exist_ok=True)   # PBF files
DB_DIR       = Path('../db');   DB_DIR.mkdir(exist_ok=True)    # DuckDB files
BOUNDARY_DIR = Path('../boundaries')

COUNTRY_PBF   = MAP_DIR / Path(COUNTRY_URL).name   # e.g. map/sweden-latest.osm.pbf
BOUNDARY_PATH = BOUNDARY_DIR / f'{BOUNDARY_NAME}.geojson'
OUTPUT_PBF    = MAP_DIR / f'{BOUNDARY_NAME}.osm.pbf'

print(f'Country PBF   : {COUNTRY_PBF}')
print(f'Boundary      : {BOUNDARY_PATH}')
print(f'Output PBF    : {OUTPUT_PBF}')
print(f'DuckDB dir    : {DB_DIR}   (set by config, used by notebook 2)')

if not BOUNDARY_PATH.exists():
    print(f'\nWARNING: boundary file not found — run notebook 0 first')

Country PBF   : ../map/sweden-latest.osm.pbf
Boundary      : ../boundaries/nacka.geojson
Output PBF    : ../map/nacka.osm.pbf
DuckDB dir    : ../db   (set by config, used by notebook 2)
CPU times: user 1.8 s, sys: 164 ms, total: 1.96 s
Wall time: 708 ms


---
## Boundary preview

Verify the boundary before downloading — a wrong boundary means filtering the wrong area.

In [2]:
%%time
gdf    = gpd.read_file(BOUNDARY_PATH).to_crs('EPSG:4326')
bounds = gdf.total_bounds
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

m = folium.Map(location=center, zoom_start=12, tiles='OpenStreetMap')
folium.GeoJson(
    gdf.__geo_interface__,
    style_function=lambda _: {'color': 'navy', 'weight': 2, 'fillOpacity': 0.15},
    tooltip=BOUNDARY_NAME,
).add_to(m)
m

CPU times: user 3.33 ms, sys: 30.3 ms, total: 33.6 ms
Wall time: 90.1 ms


---
## Download country PBF

The country PBF is downloaded to `map/` and cached — if the file already exists
this step is skipped instantly.

File sizes vary by country:

| Country | Approx size |
|---|---|
| Sweden | ~780 MB |
| Netherlands | ~160 MB |
| Germany | ~4 GB |
| California (US) | ~900 MB |

Download speed depends on your connection — Geofabrik servers are fast (usually 5–20 MB/s).

In [3]:
%%time
if COUNTRY_PBF.exists():
    size_mb = COUNTRY_PBF.stat().st_size / 1_048_576
    print(f'Already cached: {COUNTRY_PBF.name}  ({size_mb:.0f} MB) — skipping download')
else:
    print(f'Downloading {COUNTRY_URL} ...')
    print('This may take a few minutes depending on country size and connection speed.')
    print()

    with requests.get(COUNTRY_URL, stream=True, timeout=30) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        downloaded = 0
        with open(COUNTRY_PBF, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):  # 8 MB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = downloaded / total * 100
                    print(f'  {downloaded/1_048_576:.0f} / {total/1_048_576:.0f} MB  ({pct:.0f}%)', end='\r')

    size_mb = COUNTRY_PBF.stat().st_size / 1_048_576
    print(f'\nDownloaded: {COUNTRY_PBF.name}  ({size_mb:.0f} MB)')

Already cached: sweden-latest.osm.pbf  (768 MB) — skipping download
CPU times: user 0 ns, sys: 646 μs, total: 646 μs
Wall time: 378 μs


---
## Filter PBF by boundary

`osmium extract` reads the country PBF and writes a new one containing only features
that intersect the boundary polygon.

The `--overwrite` flag allows re-running without deleting the output first.

**Typical output sizes:**
- Small district (Södermalm ~10 km²) → ~1 MB
- Municipality (Nacka ~180 km²) → ~5 MB

In [4]:
%%time
print(f'Filtering {COUNTRY_PBF.name} → {OUTPUT_PBF.name} ...')
cmd = [
    'osmium', 'extract',
    '--polygon', str(BOUNDARY_PATH.resolve()),
    '--output',  str(OUTPUT_PBF.resolve()),
    '--overwrite',
    str(COUNTRY_PBF.resolve()),
]
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print('ERROR:', result.stderr)
    raise RuntimeError('osmium extract failed — is osmium-tool installed?')

size_mb = OUTPUT_PBF.stat().st_size / 1_048_576
print(f'Done: {OUTPUT_PBF.name}  ({size_mb:.2f} MB)')
print()
print('Next step: open notebook 2_build_network.ipynb')
print(f'  Set NAME = "{BOUNDARY_NAME}"')

Filtering sweden-latest.osm.pbf → nacka.osm.pbf ...
Done: nacka.osm.pbf  (3.39 MB)

Next step: open notebook 2_build_network.ipynb
  Set NAME = "nacka"
CPU times: user 0 ns, sys: 2.43 ms, total: 2.43 ms
Wall time: 4.04 s


---
## Generate duckOSM config file

duckOSM reads a YAML config file that tells it where the PBF is, where to write the
DuckDB, and which processing options to use. We generate it here automatically so
notebook 2 can just pass the config path to `DuckOSM()` without any manual editing.

The config is saved to `config/{boundary_name}.yaml` with **absolute paths** —
duckOSM requires absolute paths to locate files correctly regardless of where it is run from.

**Options you can tune:**
| Option | Default | Effect |
|---|---|---|
| `h3_resolution` | 8 | H3 spatial index resolution — match notebook 5 |
| `simplify` | true | Contract degree-2 nodes (smaller DB, faster routing) |
| `modes` | driving, walking, cycling | Transportation modes to process |

In [5]:
%%time
CONFIG_DIR = Path('../config');  CONFIG_DIR.mkdir(exist_ok=True)
CONFIG_PATH = CONFIG_DIR / f'{BOUNDARY_NAME}.yaml'

# ── duckOSM options — edit if needed ─────────────────────────────────────
H3_RESOLUTION = 8
MODES         = ['driving', 'walking', 'cycling']
# ─────────────────────────────────────────────────────────────────────────

config_content = f"""name: "{BOUNDARY_NAME}"
pbf_path: "{OUTPUT_PBF.resolve()}"
output_path: "{DB_DIR.resolve()}"
# → Creates: {(DB_DIR / BOUNDARY_NAME).resolve()}.duckdb

boundary_path: "{BOUNDARY_PATH.resolve()}"

options:
  build_graph: true          # edge adjacency table for routing
  h3_indexing: true          # H3 spatial index on edges
  h3_resolution: {H3_RESOLUTION}            # H3 resolution (0-15) — match notebook 5
  simplify: true             # contract degree-2 nodes (smaller DB, faster routing)
  process_speeds: true       # normalise speed limits
  extract_restrictions: true # extract turn restrictions
  calculate_costs: true      # calculate travel time in seconds

modes:
{chr(10).join(f"  - {m}" for m in MODES)}
"""

CONFIG_PATH.write_text(config_content)
print(f'Config written → {CONFIG_PATH}')
print()
print(config_content)

Config written → ../config/nacka.yaml

name: "nacka"
pbf_path: "/home/kaveh/projects/osm-traffic-enrichment/map/nacka.osm.pbf"
output_path: "/home/kaveh/projects/osm-traffic-enrichment/db"
# → Creates: /home/kaveh/projects/osm-traffic-enrichment/db/nacka.duckdb

boundary_path: "/home/kaveh/projects/osm-traffic-enrichment/boundaries/nacka.geojson"

options:
  build_graph: true          # edge adjacency table for routing
  h3_indexing: true          # H3 spatial index on edges
  h3_resolution: 8            # H3 resolution (0-15) — match notebook 5
  simplify: true             # contract degree-2 nodes (smaller DB, faster routing)
  process_speeds: true       # normalise speed limits
  extract_restrictions: true # extract turn restrictions
  calculate_costs: true      # calculate travel time in seconds

modes:
  - driving
  - walking
  - cycling

CPU times: user 0 ns, sys: 1.88 ms, total: 1.88 ms
Wall time: 1.62 ms
